# Mini Transformer for Saathi intent classification

Sairaj's Day 34 exercise. This encoder classifier is trained from scratch on synthetic intent data. It demonstrates attention; it is not a production language model. The training implementation is shared with `ml/mini_transformer.py` so tests exercise the same code.

In [1]:
from pathlib import Path
import sys
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'ml' / 'mini_transformer.py').exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
import torch
from ml.common import ARTIFACTS, LABELS, load_split
from ml.mini_transformer import MultiHeadAttention, MiniTransformer, encode, make_vocab
torch.manual_seed(42)
torch.set_num_threads(4)
train = load_split('train')
vocab = make_vocab(train.text)
print('Training examples:', len(train), 'Vocabulary:', len(vocab))

Training examples: 1000 Vocabulary: 80


## Scaled dot product attention and multiple heads

Each head computes $\mathrm{softmax}(QK^T/\sqrt{d_k})V$. Keys at padding positions receive negative infinity before softmax. Four heads learn separate projections; their outputs are concatenated and projected back to the embedding width.

In [2]:
import inspect
print(inspect.getsource(MultiHeadAttention))
model = MiniTransformer(len(vocab)).eval()
ids = encode(['Compare Silver and Gold for C-204'], vocab)
with torch.no_grad():
    logits, attention = model(ids)
print('Logits:', tuple(logits.shape))
print('Attention: batch, heads, query, key =', tuple(attention.shape))
assert torch.all(attention.masked_select(~ids.ne(0)[:, None, None, :]) == 0)
print('Padding attention is exactly zero.')

class MultiHeadAttention(nn.Module):
    def __init__(self, width=64, heads=4):
        super().__init__()
        if width % heads:
            raise ValueError("Embedding width must be divisible by heads")
        self.heads, self.dim = heads, width // heads
        self.qkv = nn.Linear(width, width*3)
        self.output = nn.Linear(width, width)

    def forward(self, x, mask):
        batch, length, width = x.shape
        q, k, v = self.qkv(x).reshape(batch, length, 3, self.heads, self.dim).permute(2,0,3,1,4).unbind(0)
        logits = q @ k.transpose(-1,-2) / math.sqrt(self.dim)
        logits = logits.masked_fill(~mask[:,None,None,:], float("-inf"))
        weights = logits.softmax(-1)
        attended = (weights @ v).transpose(1,2).reshape(batch,length,width)
        return self.output(attended), weights

Logits: (1, 5)
Attention: batch, heads, query, key = (1, 4, 40, 40)
Padding attention is exactly zero.


## Positional encoding and encoder classifier

Sinusoidal positions use $PE_{pos,2i}=\sin(pos/10000^{2i/d})$ and $PE_{pos,2i+1}=\cos(pos/10000^{2i/d})$. Residual connections, layer normalization, and a feed-forward network form one encoder block. Masked mean pooling feeds a five-class output layer. There is no autoregressive decoder because the task predicts a class, not a generated sequence.

In [3]:
print(inspect.getsource(MiniTransformer))
print('First positions, first eight dimensions:')
print(model.position[:4, :8])
print('Parameters:', sum(p.numel() for p in model.parameters()))

class MiniTransformer(nn.Module):
    def __init__(self, vocab_size, classes=5, width=64, heads=4, length=40):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, width, padding_idx=0)
        position = torch.arange(length).float().unsqueeze(1)
        divisor = torch.exp(torch.arange(0,width,2).float()*(-math.log(10000.0)/width))
        pe = torch.zeros(length,width)
        pe[:,0::2], pe[:,1::2] = torch.sin(position*divisor), torch.cos(position*divisor)
        self.register_buffer("position", pe)
        self.attention = MultiHeadAttention(width,heads)
        self.norm1, self.norm2 = nn.LayerNorm(width), nn.LayerNorm(width)
        self.feedforward = nn.Sequential(nn.Linear(width,128), nn.GELU(), nn.Linear(128,width))
        self.classifier = nn.Linear(width,classes)

    def forward(self, ids):
        mask = ids.ne(0)
        if not mask.any(dim=1).all():
            raise ValueError("Each input must contain a non-padding token")
        x = self.embe

## Training and held-out result

Run `python -m ml.mini_transformer` from the project root to reproduce 30 epochs. The vocabulary is fit only on training text, the checkpoint is selected using the development set, and the locked test set is evaluated afterward. Do not tune using the test score; new model iterations need new independent test data.

In [4]:
import json
metrics = json.loads((ARTIFACTS / 'mini_transformer' / 'metrics.json').read_text())
history = json.loads((ARTIFACTS / 'mini_transformer' / 'history.json').read_text())
print('Held-out macro F1:', metrics['macro_f1'])
print('Best development macro F1:', metrics['metadata']['best_dev_macro_f1'])
print('Last epoch:', history[-1])
print('Confusion matrix label order:', metrics['label_order'])
for row in metrics['confusion_matrix']:
    print(row)

Held-out macro F1: 0.43698191986568846
Best development macro F1: 0.697041049717106
Last epoch: {'epoch': 30, 'loss': 0.00012473868991946802, 'dev_macro_f1': 0.6882107491826162}
Confusion matrix label order: ['compare_products', 'coverage_question', 'renewal_draft', 'customer_lookup', 'out_of_scope']
[6, 24, 0, 0, 0]
[0, 15, 0, 0, 15]
[0, 0, 4, 12, 14]
[0, 13, 0, 17, 0]
[0, 0, 0, 0, 30]


## Limitations

Template-family holdout is stricter than a random row split, but these sentences are still synthetic. A tiny encoder trained on 1,000 examples may memorize vocabulary and fail on paraphrases. Compare its actual score with TF-IDF, pretrained zero-shot NLI, and fine-tuned DistilBERT in the experiment report. The notebook does not claim real-world accuracy or fairness.